# Apple Music Matching Pipeline
## Notebook Summary

This notebook matches Billboard Top-40 songs to Apple Music metadata using a custom similarity-scoring system. Match quality is validated through automated thresholds and manual review, producing a curated dataset of songs with Apple metadata and preview URLs.

### Inputs

* `top40_billboard_dataset.csv`

### Outputs

* `top40_apple_matches.csv`
* `top40_apple_matches_v2_reviewed.csv`
* `usable_apple_matches_v2.csv`

### Key Contributions

* Develop a similarity-based Apple Music matching pipeline.
* Validate and review low-confidence matches.
* Generate the Apple metadata dataset used for audio feature extraction.



## 0) Setup + Load

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import re
import os

from difflib import SequenceMatcher
from pathlib import Path

# File paths
TOP40_PATH = "../data/intermediate/top40_billboard_dataset.csv"

APPLE_MATCHES_PATH = "../data/intermediate/top40_apple_matches.csv"
APPLE_REVIEWED_PATH = "../data/intermediate/top40_apple_matches_v2_reviewed.csv"
USABLE_APPLE_PATH = "../data/intermediate/usable_apple_matches_v2.csv"

FLAGGED_REVIEW_PATH = "../data/review/flagged_apple_matches_for_review.csv"
LIKELY_RECOVERABLE_PATH = "../data/review/likely_recoverable_flagged_matches.csv"
FORMATTING_RECOVERY_PATH = "../data/review/formatting_recovery_candidates.csv"

In [5]:
# Load Top-40 Billboard dataset generated in Notebook 1
top40 = pd.read_csv(TOP40_PATH)

print("Top-40 dataset shape:", top40.shape)

Top-40 dataset shape: (4078, 17)


## 1) Apple Search + Matching Logic

- Implements text cleaning, main-artist extraction, similarity scoring, and version-penalty helpers used to evaluate candidate tracks.
- Queries the iTunes Search API, scores results via score_match and search_itunes_best_match, and classifies matches (AUTO_ACCEPT / REVIEW / LOW_CONFIDENCE) based on thresholds.
- Provides a retry/backoff wrapper and an artist_overlap utility to support batch matching, filtering, and downstream usability decisions.

In [ ]:
# Clean text for better matching
def clean_text(text):
    text = str(text).lower()
    text = text.replace('"', "")
    text = text.replace("'", "")
    text = re.sub(r"\(.*?\)", "", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# Extract the main artist from a string
def get_main_artist(artist):
    artist = str(artist)
    artist = re.split(
        r" featuring | feat\. | feat | ft\. | ft | & | and |,",
        artist,
        flags=re.IGNORECASE
    )[0]
    return artist.strip()

In [ ]:
# Calculate similarity between two strings
def similarity(a, b):
    return SequenceMatcher(None, clean_text(a), clean_text(b)).ratio()

In [ ]:
bad_version_words = [
    "piano", "acoustic", "live", "remix", "karaoke",
    "instrumental", "sped up", "slowed", "cover",
    "tribute", "demo", "reimagined", "stripped",
    "version", "tribute", "originally performed",
    "kidz bop", "16 bit", "computer game"
]


# cleaner that PRESERVES words inside parentheses/brackets
# so things like "(Acoustic)" don't disappear

def simple_clean(text):
    text = str(text).lower()
    text = text.replace('"', "")
    text = text.replace("'", "")

    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def version_penalty(track_name, collection_name):
    
    text = simple_clean(
        str(track_name) + " " + str(collection_name)
    )

    penalty = 0

    for word in bad_version_words:
        if word in text:
            penalty += 0.25

    return penalty

In [ ]:
# Score a match based on title, artist, and presence of preview URL
def score_match(search_song, search_artist, result):
    matched_song = result.get("trackName", "")
    matched_artist = result.get("artistName", "")
    collection = result.get("collectionName", "")
    preview_url = result.get("previewUrl", None)
    
    title_score = similarity(search_song, matched_song)
    artist_score = similarity(get_main_artist(search_artist), matched_artist)
    penalty = version_penalty(matched_song, collection)
    
    preview_bonus = 0.05 if preview_url else -0.30
    
    final_score = (
        0.65 * title_score
        + 0.30 * artist_score
        + preview_bonus
        - penalty
    )
    
    return final_score, title_score, artist_score, penalty

In [ ]:
# Search Apple/iTunes API, score possible matches, and return the best result.
def search_itunes_best_match(song, artist):
    """
    Search Apple/iTunes API, score possible matches, and return the best result.
    Also collects Apple metadata:
    - preview_url
    - release date
    - duration
    - genre
    """

    url = "https://itunes.apple.com/search"

    search_term = f"{clean_text(song)} {clean_text(get_main_artist(artist))}"

    params = {
        "term": search_term,
        "media": "music",
        "entity": "song",
        "limit": 10
    }

    try:
        response = requests.get(url, params=params, timeout=10)

    except requests.exceptions.RequestException:
        return {
            "search_song": song,
            "search_artist": artist,
            "search_term": search_term,
            "failure_reason": "REQUEST_ERROR"
        }

    if response.status_code != 200:
        return {
            "search_song": song,
            "search_artist": artist,
            "search_term": search_term,
            "failure_reason": "BAD_STATUS",
            "status_code": response.status_code
        }

    try:
        data = response.json()

    except ValueError:
        return {
            "search_song": song,
            "search_artist": artist,
            "search_term": search_term,
            "failure_reason": "JSON_ERROR",
            "response_preview": response.text[:200]
        }

    if data.get("resultCount", 0) == 0:
        return {
            "search_song": song,
            "search_artist": artist,
            "search_term": search_term,
            "failure_reason": "NO_RESULTS"
        }

    scored_results = []

    for result in data["results"]:
        score, title_score, artist_score, penalty = score_match(song, artist, result)
        scored_results.append((score, title_score, artist_score, penalty, result))

    best_score, title_score, artist_score, penalty, best_result = max(
        scored_results,
        key=lambda x: x[0]
    )

    preview_url = best_result.get("previewUrl")
    track_time_ms = best_result.get("trackTimeMillis")

    if track_time_ms is not None:
        duration_sec = track_time_ms / 1000
    else:
        duration_sec = None

    if (
        best_score >= 0.92
        and title_score >= 0.90
        and artist_score >= 0.65
        and penalty == 0
        and preview_url is not None
    ):
        match_status = "AUTO_ACCEPT"

    elif (
        best_score >= 0.80
        and title_score >= 0.90
        and artist_score >= 0.50
        and penalty == 0
        and preview_url is not None
    ):
        match_status = "REVIEW"

    else:
        match_status = "LOW_CONFIDENCE"

    return {
        # original Billboard search info
        "search_song": song,
        "search_artist": artist,
        "search_term": search_term,

        # Apple matched info
        "matched_song": best_result.get("trackName"),
        "matched_artist": best_result.get("artistName"),
        "collection": best_result.get("collectionName"),
        "preview_url": preview_url,

        # Apple metadata
        "apple_release_date": best_result.get("releaseDate"),
        "apple_track_time_ms": track_time_ms,
        "apple_duration_sec": duration_sec,
        "apple_genre": best_result.get("primaryGenreName"),
        "apple_collection": best_result.get("collectionName"),
        "apple_track_id": best_result.get("trackId"),
        "apple_artist_id": best_result.get("artistId"),

        # matching diagnostics
        "match_score": best_score,
        "title_score": title_score,
        "artist_score": artist_score,
        "version_penalty": penalty,
        "match_status": match_status,
        "failure_reason": "SUCCESS"
    }

In [ ]:
# Wrapper around search_itunes_best_match().
def search_itunes_best_match_with_backoff(
    song,
    artist,
    max_retries=3,
    base_sleep=10
):
    """
    Wrapper around search_itunes_best_match().
    If Apple returns 429/403 or a request-like failure, wait and retry.
    """

    for attempt in range(max_retries + 1):

        result = search_itunes_best_match(song, artist)

        failure_reason = result.get("failure_reason")
        status_code = result.get("status_code")

        # success or non-rate-limit failure
        if failure_reason != "BAD_STATUS":
            return result

        # rate limit / temporary block
        if status_code in [429, 403]:

            wait_time = base_sleep * (attempt + 1)

            print(
                f"Rate limit/status {status_code} for: {song} — {artist}. "
                f"Waiting {wait_time}s before retry {attempt + 1}/{max_retries}..."
            )

            time.sleep(wait_time)

        else:
            return result

    # if retries fail, return final result
    return result

In [ ]:
# Calculate artist name word overlap as an additional similarity metric
def artist_overlap(search_artist, matched_artist):
    search_clean = clean_text(search_artist)
    matched_clean = clean_text(matched_artist)

    search_words = set(search_clean.split())
    matched_words = set(matched_clean.split())

    if len(search_words) == 0 or len(matched_words) == 0:
        return 0

    overlap = search_words.intersection(matched_words)
    return len(overlap) / min(len(search_words), len(matched_words))

## 2) Matching Validation Sample (n=100)

- Sampled 100 random Top-40 songs (random_state=42) and queried iTunes for best matches.
- Collected diagnostics (match_score, title_score, artist_score, version_penalty, preview_url) and classified results as AUTO_ACCEPT / REVIEW / LOW_CONFIDENCE.
- Marked matches as usable if AUTO_ACCEPT or (title_score ≥ 0.95, artist_overlap ≥ 0.40, no version_penalty, and preview_url present).

In [ ]:
# Run a small batch of test searches to preview results before scaling
test_songs = song_life[["Song", "Artist"]].sample(
    100,
    random_state=42
).copy()

preview_results = []

for _, row in test_songs.iterrows():
    result = search_itunes_best_match(row["Song"], row["Artist"])
    preview_results.append(result)
    time.sleep(0.5)

preview_df = pd.DataFrame(preview_results)

In [ ]:
preview_df["failure_reason"].value_counts()

failure_reason
SUCCESS    100
Name: count, dtype: int64

In [ ]:
preview_df["match_status"].value_counts(dropna=False)

match_status
AUTO_ACCEPT       82
REVIEW            11
LOW_CONFIDENCE     7
Name: count, dtype: int64

In [ ]:
preview_df[
    ["search_song", "search_artist", "matched_song", "matched_artist", 
     "collection", "match_score", "match_status"]
].sample(20, random_state=1)

,search_song,search_artist,matched_song,matched_artist,collection,match_score,match_status
30,Friday Night,Eric Paslay,Friday Night,Eric Paslay,Eric Paslay,1.000000,AUTO_ACCEPT
2,Otis,Jay Z Kanye West Featuring Otis Redding,Otis (feat. Otis Redding),Kanye West & JAŸ-Z,Watch the Throne (Deluxe Version),0.893548,AUTO_ACCEPT
51,X2,Lil Uzi Vert,x2,Lil Uzi Vert,Pink Tape,1.000000,AUTO_ACCEPT
32,Beg For It,Iggy Azalea Featuring M0,Beg For It (feat. M.O),Iggy Azalea,Reclassified,1.000000,AUTO_ACCEPT
31,I'm Still A Guy,Brad Paisley,I'm Still a Guy,Brad Paisley,5th Gear (Bonus Track Version),1.000000,AUTO_ACCEPT
46,Electricity,Silk City x Dua Lipa,Electricity (feat. Dua Lipa) [Acoustic],Silk City & Dua Lipa (Diplo and Marc Ronson),Electricity (feat. Dua Lipa) [Acoustic] - Single,0.984211,AUTO_ACCEPT
34,Body Language,Jesse McCartney Featuring T-Pain,Body Language (feat. T-Pain),Jesse McCartney,Body Language (feat. T-Pain) - Single,1.000000,AUTO_ACCEPT
39,No Opp Left Behind,21 Savage & Metro Boomin,No Opp Left Behind,21 Savage & Metro Boomin,SAVAGE MODE II,0.874194,AUTO_ACCEPT
45,Social Distancing,Lil Baby,Social Distancing,Lil Baby,My Turn (Deluxe),1.000000,AUTO_ACCEPT
19,You Complete Me,Keyshia Cole,You Complete Me,Keyshia Cole,A Different Me,1.000000,AUTO_ACCEPT


In [ ]:
# Filter for auto-accepted matches
auto_accept = preview_df[
    preview_df["match_status"] == "AUTO_ACCEPT"
]

In [ ]:
auto_accept[
    [
        "search_song",
        "search_artist",
        "matched_song",
        "matched_artist",
        "match_score"
    ]
].head()

,search_song,search_artist,matched_song,matched_artist,match_score
0,"""Girl, Get Up.""",Doechii & SZA,"girl, get up.",Doechii & SZA,0.933333
1,Bright,Echosmith,Bright,Echosmith,1.000000
3,Paramedic!,SOB X RBE,Paramedic!,SOB X RBE,1.000000
4,Let It Go,Keyshia Cole Featuring Missy Elliott & Lil Kim,Let It Go (feat. Missy Elliott & Lil 'Kim),Keyshia Cole,1.000000
5,I Wrote The Book,Morgan Wallen,I Wrote The Book,Morgan Wallen,1.000000


## 3) Test on 250 random songs

- Evaluate matching performance on a larger sample before running the full dataset.
- Review match quality, confidence scores, and failure cases.
- Confirm that the matching pipeline scales reliably beyond small test samples.

In [ ]:
# take sample of 250 songs for more extensive testing before running on full dataset
test_songs = song_life[["Song", "Artist"]].sample(
    250,
    random_state=42
).copy()

preview_results = []

for _, row in test_songs.iterrows():
    result = search_itunes_best_match(row["Song"], row["Artist"])
    preview_results.append(result)
    time.sleep(1.0)

preview_df = pd.DataFrame(preview_results)

preview_df["failure_reason"].value_counts()

failure_reason
SUCCESS    250
Name: count, dtype: int64

In [ ]:
preview_df["match_status"].value_counts(dropna=False)

match_status
AUTO_ACCEPT       206
LOW_CONFIDENCE     24
REVIEW             20
Name: count, dtype: int64

In [ ]:
# calculate artist name overlap for review/low confidence matches
preview_df["artist_overlap"] = preview_df.apply(
    lambda row: artist_overlap(row["search_artist"], row["matched_artist"]),
    axis=1
)

# determine which matches are usable (either auto-accepted or high-confidence review candidates)
preview_df["usable_match"] = (
    (preview_df["match_status"] == "AUTO_ACCEPT")
    |
    (
        (preview_df["title_score"] >= 0.95)
        & (preview_df["artist_overlap"] >= 0.40)
        & (preview_df["version_penalty"] == 0)
        & (preview_df["preview_url"].notna())
    )
)

# review the low-confidence matches that are potentially usable based on title score and artist overlap
preview_df[
    (preview_df["usable_match"] == True)
    & (preview_df["match_status"] != "AUTO_ACCEPT")
][
    ["search_song", "search_artist", "matched_song", "matched_artist",
     "title_score", "artist_score", "artist_overlap", "version_penalty", "match_status"]
].sort_values("artist_overlap").head(30)

,search_song,search_artist,matched_song,matched_artist,title_score,artist_score,artist_overlap,version_penalty,match_status
137,No One Mourns The Wicked,"""Ariana Grande Featuring Andy Nyman, Courtney-...","No One Mourns the Wicked (feat. Andy Nyman, Co...",Wicked Movie Cast & Ariana Grande,1.0,0.590909,0.40,0.0,REVIEW
2,Otis,Jay Z Kanye West Featuring Otis Redding,Otis (feat. Otis Redding),Kanye West & JAŸ-Z,1.0,0.645161,0.75,0.0,REVIEW
181,Now Or Never,High School Musical 3 Cast,Now or Never,The Cast of High School Musical,1.0,0.666667,0.80,0.0,REVIEW
218,OK Not To Be OK,Marshmello & Demi Lovato,OK Not To Be OK,Marshmello & Demi Lovato,1.0,0.625000,1.00,0.0,REVIEW
216,Sharks,"""Lil Wayne, Jelly Roll & Big Sean""",Sharks,"Lil Wayne, Jelly Roll & Big Sean",1.0,0.473684,1.00,0.0,LOW_CONFIDENCE
203,Fine China,Future & Juice WRLD,Fine China,Future & Juice WRLD,1.0,0.521739,1.00,0.0,REVIEW
164,Amen,Shaboozey & Jelly Roll,Amen,Shaboozey & Jelly Roll,1.0,0.620690,1.00,0.0,REVIEW
125,Hate Me,Ellie Goulding & Juice WRLD,Hate Me,Ellie Goulding & Juice WRLD,1.0,0.717949,1.00,0.0,REVIEW
110,Santa Baby,Eartha Kitt With Henri Rene And His Orchestra,Santa Baby (with Henri René and His Orchestra),Eartha Kitt,1.0,0.578947,1.00,0.0,REVIEW
219,We Own It (Fast & Furious),2 Chainz & Wiz Khalifa,We Own It (Fast & Furious),2 Chainz & Wiz Khalifa,1.0,0.571429,1.00,0.0,REVIEW


In [ ]:
# summarize how many matches are auto-accepted, how many are usable (either auto-accepted or high-confidence review candidates), and how many additional matches we can potentially recover by reviewing the high-confidence review candidates
auto_accept_count = (
    preview_df["match_status"] == "AUTO_ACCEPT"
).sum()

usable_count = preview_df["usable_match"].sum()

print("AUTO_ACCEPT:", auto_accept_count)
print("USABLE:", usable_count)
print("Recovered:", usable_count - auto_accept_count)

AUTO_ACCEPT: 206
USABLE: 227
Recovered: 21


In [ ]:
# review the low-confidence matches that are potentially usable, excluding the auto-accepted matches
preview_df[
    preview_df["match_status"] != "AUTO_ACCEPT"
][
    ["search_song", "search_artist", "matched_song", "matched_artist",
     "collection", "match_score", "title_score", "artist_score", "version_penalty", "match_status"]
].sort_values("match_score").head(30)

,search_song,search_artist,matched_song,matched_artist,collection,match_score,title_score,artist_score,version_penalty,match_status
22,M3tamorphosis,Playboi Carti Featuring Kid Cudi,M3tamorphosis (8-Bit Computer Game Version),8-Bit Arcade,"By Request, Vol. 117",0.272000,1.000000,0.240000,0.50,LOW_CONFIDENCE
68,Working,Tate McRae X Khalid,she's all i wanna be,Tate McRae,she's all i wanna be - Single,0.356897,0.153846,0.689655,0.00,LOW_CONFIDENCE
153,Dis 1 Got It,Playboi Carti,Let It Go (Frozen),Lullaby Baby Trio,Disney Lullabies Classic Renditions of Disney ...,0.397619,0.380952,0.333333,0.00,LOW_CONFIDENCE
83,You Got Em,Lil Durk,All My Life (feat. J. Cole),Lil Durk,All My Life (feat. J. Cole) - Single,0.535714,0.285714,1.000000,0.00,LOW_CONFIDENCE
226,F**k The Industry Pt. 2,YoungBoy Never Broke Again,Make No Sense,YoungBoy Never Broke Again,AI YoungBoy 2,0.541176,0.294118,1.000000,0.00,LOW_CONFIDENCE
186,Untouchable,YoungBoy Never Broke Again,Slime Mentality,YoungBoy Never Broke Again,AI YoungBoy 2,0.550000,0.307692,1.000000,0.00,LOW_CONFIDENCE
193,Outfit,Lil Baby & 21 Savage,Close Friends,Lil Baby,Drip Harder,0.555263,0.315789,1.000000,0.00,LOW_CONFIDENCE
11,Got The Guap,Lil Uzi Vert Featuring Young Thug,Sauce It Up,Lil Uzi Vert,Luv Is Rage 2,0.576087,0.347826,1.000000,0.00,LOW_CONFIDENCE
182,Cold Heart (PNAU Remix),Elton John & Dua Lipa,Cold Heart (PNAU Remix),Elton John & Dua Lipa,The Lockdown Sessions,0.656897,1.000000,0.689655,0.25,LOW_CONFIDENCE
53,Anyways,Zach Bryan,Always Willin',Zach Bryan,With Heaven On Top,0.675000,0.500000,1.000000,0.00,LOW_CONFIDENCE


## 4) Full Top-40 Apple Matching Job

- Apply the validated matching pipeline to all Top-40 Billboard songs.
- Retrieve Apple metadata and preview URLs using batch processing and checkpointing.
- Generate the complete Apple match dataset for downstream audio analysis.

In [ ]:
output_file = "../data/intermediate/top40_apple_matches.csv"

batch_size = 100
sleep_between_requests = 1.0
sleep_between_batches = 10

In [ ]:
# Load existing progress if file exists

if os.path.exists(output_file):
    existing_results = pd.read_csv(output_file)

    already_done = set(
        zip(
            existing_results["search_song"],
            existing_results["search_artist"]
        )
    )

    results = existing_results.to_dict("records")

    print("Loaded existing results:", len(results))

else:
    already_done = set()
    results = []

    print("Starting fresh")

Starting fresh


In [ ]:
# Prepare songs remaining

top40_pairs = top40[
    ["Song", "Artist"]
].drop_duplicates().copy()

remaining = top40_pairs[
    ~top40_pairs.apply(
        lambda row: (
            row["Song"],
            row["Artist"]
        ) in already_done,
        axis=1
    )
].copy()

print("Total Top 40 songs:", len(top40_pairs))
print("Already done:", len(already_done))
print("Remaining:", len(remaining))

Total Top 40 songs: 4078
Already done: 0
Remaining: 4078


In [ ]:
# Run in batches with checkpointing and status reports

for batch_num, start in enumerate(
    range(0, len(remaining), batch_size),
    start=1
):

    batch = remaining.iloc[
        start:start + batch_size
    ]

    print("\n" + "=" * 70)
    print(f"Starting batch {batch_num}")
    print(f"Batch rows: {start} to {start + len(batch) - 1}")
    print("=" * 70)

    batch_results = []

    for _, row in tqdm(
        batch.iterrows(),
        total=len(batch)
    ):

        result = search_itunes_best_match_with_backoff(
            row["Song"],
            row["Artist"],
            max_retries=3,
            base_sleep=10
        )

        results.append(result)
        batch_results.append(result)

        time.sleep(sleep_between_requests)

    # Convert batch + full results to dataframes
    batch_df = pd.DataFrame(batch_results)
    results_df = pd.DataFrame(results)

    # Save checkpoint
    results_df.to_csv(
        output_file,
        index=False
    )

    # Batch-level status report
    print("\nBatch saved.")
    print("Total saved rows:", len(results_df))

    print("\nBatch failure reasons:")
    print(
        batch_df["failure_reason"]
        .value_counts(dropna=False)
    )

    if "status_code" in batch_df.columns:
        print("\nBatch status codes:")
        print(
            batch_df["status_code"]
            .value_counts(dropna=False)
        )

    if "match_status" in batch_df.columns:
        print("\nBatch match statuses:")
        print(
            batch_df["match_status"]
            .value_counts(dropna=False)
        )

    # Quick usable-match estimate for this batch
    if set([
        "match_status",
        "title_score",
        "version_penalty",
        "preview_url",
        "search_artist",
        "matched_artist"
    ]).issubset(batch_df.columns):

        batch_df["artist_overlap"] = batch_df.apply(
            lambda row: artist_overlap(
                row["search_artist"],
                row["matched_artist"]
            )
            if row["failure_reason"] == "SUCCESS"
            else None,
            axis=1
        )

        batch_df["usable_match_temp"] = (
            (batch_df["match_status"] == "AUTO_ACCEPT")
            |
            (
                (batch_df["title_score"] >= 0.95)
                & (batch_df["artist_overlap"] >= 0.40)
                & (batch_df["version_penalty"] == 0)
                & (batch_df["preview_url"].notna())
            )
        )

        print("\nBatch usable match estimate:")
        print(
            batch_df["usable_match_temp"]
            .value_counts(dropna=False)
        )

        print(
            "Batch usable rate:",
            round(batch_df["usable_match_temp"].mean(), 3)
        )

    # Overall status report
    print("\nOverall failure reasons:")
    print(
        results_df["failure_reason"]
        .value_counts(dropna=False)
    )

    if "match_status" in results_df.columns:
        print("\nOverall match statuses:")
        print(
            results_df["match_status"]
            .value_counts(dropna=False)
        )

    if start + batch_size < len(remaining):
        print(f"\nResting {sleep_between_batches}s before next batch...")
        time.sleep(sleep_between_batches)

### Full run summary

The full Apple matching job was run on the Top-40 Billboard dataset.

- Input songs: 4,078
- Batch size: 250
- Sleep interval: 1 second
- Output: `top40_apple_matches.csv`
- Runtime: approximately 1.5 hours
- Result: Apple metadata and preview URLs were successfully retrieved for the majority of songs.

## 5) Match Quality Review

- Evaluate overall matching success and failure rates.
- Apply automated quality thresholds to identify usable matches.
- Quantify confidence levels before manual review and recovery.

In [ ]:
# Load the full results and calculate artist overlap and usable matches for final review and filtering
apple_df = pd.read_csv(
    "../data/intermediate/top40_apple_matches.csv"
)

apple_df["artist_overlap"] = apple_df.apply(
    lambda row: artist_overlap(
        row["search_artist"],
        row["matched_artist"]
    )
    if row["failure_reason"] == "SUCCESS"
    else None,
    axis=1
)

apple_df["usable_match"] = (
    (apple_df["match_status"] == "AUTO_ACCEPT")
    |
    (
        (apple_df["title_score"] >= 0.95)
        & (apple_df["artist_overlap"] >= 0.40)
        & (apple_df["version_penalty"] == 0)
        & (apple_df["preview_url"].notna())
    )
)

apple_df.to_csv(
    "../data/intermediate/top40_apple_matches.csv",
    index=False
)

In [ ]:
apple_df["match_status"].value_counts(dropna=False)

match_status
AUTO_ACCEPT       3359
LOW_CONFIDENCE     471
REVIEW             240
NaN                  8
Name: count, dtype: int64

In [ ]:
apple_df["usable_match"].value_counts(dropna=False)

usable_match
True     3675
False     403
Name: count, dtype: int64

In [ ]:
apple_df["usable_match"].mean()

np.float64(0.9011770475723394)

## 6) Formatting Recovery

- Export low-confidence matches for manual review.
- Recover likely valid matches rejected due to formatting or artist-name differences.
- Save the reviewed Apple metadata dataset used for preview downloading.

In [ ]:
flagged = apple_df[
    apple_df["usable_match"] == False
]

flagged.shape

(403, 22)

In [ ]:
review_cols = [
    "search_song",
    "search_artist",
    "matched_song",
    "matched_artist",
    "collection",
    "apple_genre",
    "preview_url",
    "match_status",
    "title_score",
    "artist_score",
    "artist_overlap",
    "version_penalty",
    "failure_reason"
]

flagged_review = flagged[review_cols].copy()

flagged_review.to_csv(
    "../data/intermediate/flagged_apple_matches_review_clean.csv",
    index=False
)

flagged_review.shape

(403, 13)

In [ ]:
flagged_review["title_score"].describe()

count    395.000000
mean       0.849279
std        0.260743
min        0.095238
25%        0.795349
50%        1.000000
75%        1.000000
max        1.000000
Name: title_score, dtype: float64

In [ ]:
flagged_review["match_status"].value_counts(dropna=False)

match_status
LOW_CONFIDENCE    378
REVIEW             17
NaN                 8
Name: count, dtype: int64

In [ ]:
likely_recoverable = flagged_review[
    (flagged_review["title_score"] >= 0.95)
    & (flagged_review["artist_overlap"] >= 0.40)
    & (flagged_review["preview_url"].notna())
].copy()

likely_recoverable.shape

In [ ]:
likely_recoverable.to_csv(
    "../data/intermediate/likely_recoverable_flagged_matches.csv",
    index=False
)

In [ ]:
print("apple_df:", apple_df.shape)

print(
    apple_df["match_status"]
    .value_counts(dropna=False)
)

print()

flagged = apple_df[
    apple_df["usable_match"] == False
]

print("flagged:", flagged.shape)

print(
    flagged["match_status"]
    .value_counts(dropna=False)
)

print()

print("flagged_review:", flagged_review.shape)

print(
    flagged_review["match_status"]
    .value_counts(dropna=False)
)

apple_df: (4078, 22)
match_status
AUTO_ACCEPT       3359
LOW_CONFIDENCE     471
REVIEW             240
NaN                  8
Name: count, dtype: int64

flagged: (403, 22)
match_status
LOW_CONFIDENCE    378
REVIEW             17
NaN                 8
Name: count, dtype: int64

flagged_review: (403, 13)
match_status
LOW_CONFIDENCE    378
REVIEW             17
NaN                 8
Name: count, dtype: int64


In [ ]:
likely_recoverable = pd.read_csv(
    "likely_recoverable_flagged_matches.csv"
)

recover_keys = set(
    zip(
        likely_recoverable["search_song"],
        likely_recoverable["search_artist"]
    )
)

rule1_mask = apple_df.apply(
    lambda row: (row["search_song"], row["search_artist"]) in recover_keys,
    axis=1
)

apple_df.loc[rule1_mask, "usable_match_v2"] = True
apple_df.loc[rule1_mask, "recovery_reason"] = "likely_recoverable_manual_rule"

apple_df["usable_match_v2"].value_counts(dropna=False)

usable_match_v2
True     3749
False     329
Name: count, dtype: int64

In [ ]:
still_flagged = apple_df[
    apple_df["usable_match_v2"] == False
].copy()

formatting_candidates = still_flagged[
    (still_flagged["title_score"] >= 0.70)
    & (still_flagged["artist_overlap"] >= 0.90)
    & (still_flagged["preview_url"].notna())
].copy()

formatting_candidates[
    [
        "search_song",
        "search_artist",
        "matched_song",
        "matched_artist",
        "collection",
        "match_status",
        "title_score",
        "artist_overlap",
        "version_penalty"
    ]
].to_csv(
    "../data/intermediate/formatting_recovery_candidates.csv",
    index=False
)

formatting_candidates.shape

(35, 24)

In [ ]:
formatting_candidates_clean = formatting_candidates[
    ~(
        (formatting_candidates["search_song"] == "East Side Of Sorrow")
        &
        (formatting_candidates["search_artist"] == "Zach Bryan")
    )
].copy()

formatting_candidates_clean.shape

(34, 24)

In [ ]:
formatting_keys = set(
    zip(
        formatting_candidates_clean["search_song"],
        formatting_candidates_clean["search_artist"]
    )
)

rule2_mask = apple_df.apply(
    lambda row: (row["search_song"], row["search_artist"]) in formatting_keys,
    axis=1
)

apple_df.loc[rule2_mask, "usable_match_v2"] = True
apple_df.loc[rule2_mask, "recovery_reason"] = "title_artist_formatting_recovery"

apple_df["usable_match_v2"].value_counts(dropna=False)

usable_match_v2
True     3783
False     295
Name: count, dtype: int64

## 7) Manual Validation Sample

- Perform manual inspection of accepted matches.
- Verify that automated scoring and recovery procedures produced accurate results.
- Identify any remaining false positives prior to final export.

In [ ]:
accepted_sample = apple_df[
    apple_df["usable_match_v2"] == True
].sample(
    100,
    random_state=42
)

accepted_sample[
    [
        "search_song",
        "search_artist",
        "matched_song",
        "matched_artist",
        "collection",
        "match_status",
        "title_score",
        "artist_overlap",
        "version_penalty"
    ]
].to_csv(
    "../data/intermediate/accepted_match_quality_sample.csv",
    index=False
)

## 8) Final Apple Metadata Export

In [ ]:
apple_df.to_csv(
    "../data/intermediate/top40_apple_matches_v2_reviewed.csv",
    index=False
)

In [ ]:
usable_apple_df = apple_df[
    apple_df["usable_match_v2"] == True
].copy()

usable_apple_df.to_csv(
    "../data/intermediate/usable_apple_matches_v2.csv",
    index=False
)

usable_apple_df.shape

(3783, 24)